# SBF-2: обрезка после нормировки

Этот ноутбук **не перестраивает галактики**. Модель, окончательная маска источников, два кольца, ансамбль PSF и $P_r$ читаются из завершённых результатов `sbf-2`. Повторяется только спектральная часть.

Сравниваются четыре ветви:

1. `no_winsor` — без ограничения пикселей; ближе всего к явно описанной методике Jensen.
2. `raw_global_3p5` — точное воспроизведение текущего production: ограничение сырых остатков, затем деление на $\sqrt M$.
3. `normalized_full_3p5` — сначала деление на $\sqrt M$, но порог всё ещё оценивается по всей области модели. Это чистая проверка порядка операций.
4. `normalized_union_3p5` — сначала деление на $\sqrt M$, затем один общий порог по объединению двух рабочих колец. Это основной кандидат.

Под «обрезкой» здесь понимается винзорирование: крайние значения прижимаются к границе, а пиксели не удаляются из окна. Маска, $E(k)$, $k$-окна, PSF и $P_r$ во всех ветвях одинаковы. Поэтому различия относятся именно к порядку обработки остаточного кадра.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

code_dir = Path.cwd() / "code"
if (code_dir / "sbf2_normalized_winsor_core.py").is_file():
    sys.path.insert(0, str(code_dir))

from sbf2_normalized_winsor_core import (
    ExperimentConfig, build_evaluation_table, discover_galaxies,
    evaluation_criteria, find_project_root, inspect_sources,
    load_matching_results, prepare_target_status,
    load_result_tables, plot_normalized_inputs, plot_power_comparison,
    process_target, target_status_path, update_target_status,
    write_aggregate_tables,
)

## 1. Конфигурация

По умолчанию берём три показательных объекта, для которых уже существует проверенный кэш $E(k)$: почти идеальный остаток, обычный случай и структурно тяжёлую галактику. NGC 4649 можно добавить четвёртой, но для неё $E(k)$ будет рассчитан с нуля. После пилота достаточно заменить `GALAXIES` на `ALL_GALAXIES`.

In [ ]:
PROJECT_ROOT = find_project_root()
SOURCE_BATCH_ROOT = PROJECT_ROOT / "runs" / "sbf2_go3055" / "batch"
OUTPUT_ROOT = PROJECT_ROOT / "runs" / "sbf2_normalized_winsor"

ALL_GALAXIES = discover_galaxies(SOURCE_BATCH_ROOT)
GALAXIES = ["NGC 3379", "NGC 1380", "NGC 4486"]
# GALAXIES = ["NGC 3379", "NGC 1380", "NGC 4649", "NGC 4486"]
# GALAXIES = ALL_GALAXIES

CONFIG = ExperimentConfig(
    normalized_sigma=3.5,
    kmins=(0.01, 0.03, 0.04),
    kmax=0.25,
    e_realizations=64,
    random_seed=1489,
    fft_workers=-1,
    save_ring_fft_fits=False,  # True добавит тяжёлые FITS колец и FFT
    save_all_branch_fits=False,  # True сохранит FITS всех четырёх ветвей
)

FORCE = False
REBUILD_INPUT_CACHE = False
REBUILD_E_CACHE = False

pd.DataFrame({"galaxy": GALAXIES})

## 2. Проверка входов

Проверяются только наличие успешного production-result и всех обязательных кэшированных продуктов. Большие FITS здесь ещё не читаются.

In [ ]:
readiness = inspect_sources(GALAXIES, SOURCE_BATCH_ROOT)
display(readiness)
READY = bool(readiness["ready"].all())
print("Все входы готовы." if READY else "Есть отсутствующие входы; см. таблицу выше.")

## 3. Спектральный пересчёт

На первом проходе для каждой галактики создаётся компактный кэш только из двух crops. Затем один раз считается и кэшируется $E(k)$ для каждого кольца и каждой PSF. Для NGC 3379, NGC 1380 и NGC 4486 уже готовый per-PSF $E(k)$ из `sbf2_systematics` используется только после полного совпадения fingerprint входов и успешного production-closure. Повторный запуск с теми же входами и параметрами читает готовый result; смена только $\sigma$ сохраняет кэш $E(k)$.

In [ ]:
results = []
failures = []
status_table, _ = prepare_target_status(
    GALAXIES, OUTPUT_ROOT, CONFIG, SOURCE_BATCH_ROOT
)
print(f"Resume table: {target_status_path(OUTPUT_ROOT)}")
display(status_table[["galaxy", "status", "stage", "attempt"]])

if READY:
    for number, galaxy in enumerate(GALAXIES, start=1):
        print(f"[{number}/{len(GALAXIES)}] {galaxy}")
        try:
            result = process_target(
                galaxy, SOURCE_BATCH_ROOT, OUTPUT_ROOT, CONFIG,
                force=FORCE,
                rebuild_input_cache=REBUILD_INPUT_CACHE,
                rebuild_expectation_cache=REBUILD_E_CACHE,
            )
            results.append(result)
            print(f"  closure={result['closure_passed']}; {result['run_dir']}")
        except KeyboardInterrupt:
            update_target_status(
                galaxy, "interrupted", OUTPUT_ROOT, CONFIG,
                stage="interrupted", message="KeyboardInterrupt",
            )
            print(f"  Прервано; состояние: {target_status_path(OUTPUT_ROOT)}")
            raise
        except Exception as error:
            failures.append({"galaxy": galaxy, "error": repr(error)})
            update_target_status(
                galaxy, "failed", OUTPUT_ROOT, CONFIG,
                stage="failed", message=repr(error),
            )
            print(f"  ОШИБКА: {error}")
else:
    print("Расчёт пропущен: сначала исправьте отсутствующие входы.")

display(pd.DataFrame(failures))
status_table, _ = prepare_target_status(
    GALAXIES, OUTPUT_ROOT, CONFIG, SOURCE_BATCH_ROOT
)
display(status_table[["galaxy", "status", "stage", "attempt", "message"]])

## 4. Численное замыкание и основная сводка

Сначала старая ветвь обязана воспроизвести production. Без этого новые сдвиги бессмысленны. Основная таблица показывает изменение $\bar m$, расхождение колец, устойчивость к выбору $k_{min}$ и долю затронутых пикселей. Все величины $\bar m$ здесь наблюдаемые; одинаковая поправка за поглощение сократится при сравнении ветвей.

In [ ]:
if results:
    campaign_results = load_matching_results(
        OUTPUT_ROOT, CONFIG, SOURCE_BATCH_ROOT
    )
    aggregate_paths = write_aggregate_tables(campaign_results, OUTPUT_ROOT)
    evaluation = build_evaluation_table(campaign_results, OUTPUT_ROOT, main_kmin=0.04)
    display(evaluation)

    closure = pd.concat([
        load_result_tables(result)["production_closure"]
        for result in results
    ], ignore_index=True)
    display(closure)

## 5. Конечные нормированные остатки

Для каждой галактики сохраняется один полноразмерный двумерный FITS. В нём уже вычтены фон и модель, применена финальная маска, остатки разделены на $\sqrt M$ и выполнено винзорирование после нормировки. Вне финальной маски стоит `NaN`, поэтому щели и вырезанные источники видны чёрными.

Это не картинка для отчёта, а численный источник основной ветви: код берёт из него пиксели кольца. Дальше остаётся только вычесть среднее выбранного кольца, выполнить FFT и подогнать $P(k)$. Среднее нельзя вычесть заранее из одного полного кадра: у внутреннего и внешнего колец оно разное.

Тяжёлые покольцевые `FFTINPUT/WINDOW` по умолчанию не пишутся. Они включаются только флагом `save_ring_fft_fits=True`. Ниже показан полный кадр; шкала симметрична относительно нуля.

In [ ]:
for result in results:
    plot_normalized_inputs(result)
    plt.show()

## 6. Что произошло со спектром мощности

Точки — измеренный $P(k)$ конкретной ветви. Линии — одна и та же модель $P_0E(k)+P_1$ при неизменном ансамбле PSF. Красивый вид линии сам по себе ничего не доказывает; важны сдвиг $\bar m$, замыкание и устойчивость между окнами $k$.

In [ ]:
for result in results:
    plot_power_comparison(result, kmin=0.04)
    plt.show()

## 7. Как принимается решение

Ветвь нельзя выбирать по самой гауссовой гистограмме или по лучшему совпадению с ожидаемым расстоянием: это подгонка. Сначала требуются техническое замыкание и физический $P_0-P_r>0$, затем устойчивость по $k$ и кольцам. Окончательное решение возможно только после теста восстановления искусственно заданного SBF-сигнала и калибровки всех 14 галактик.

In [ ]:
display(evaluation_criteria())

if results:
    clipping = pd.concat([
        load_result_tables(result)["clipping"]
        for result in results
    ], ignore_index=True)
    display(clipping[[
        "galaxy", "ring", "branch", "changed_pixels",
        "changed_fraction", "subtracted_mean",
    ]])

## 8. Восстановление искусственно заданного SBF-сигнала

Здесь известная амплитуда $P_0=0.90$ проходит через все четыре варианта винзорирования. Для каждого варианта и каждого кольца сравниваются заданное и восстановленное значения. Это проверка только порядка операций перед FFT: ошибки изофот, фона, каталога и $P_r$ намеренно не моделируются.

Главная величина — среднее смещение $\Delta\bar m$. Нулевое значение означает отсутствие обнаружимого систематического смещения. СКО показывает разброс одной реализации, а не погрешность среднего.

In [ ]:
from IPython.display import Image
from sbf2_normalized_winsor_recovery import run_recovery_test

recovery_trials, recovery_summary, recovery_figure = run_recovery_test(
    PROJECT_ROOT, OUTPUT_ROOT / "recovery", trials=64,
)

display(recovery_summary[[
    "branch_label", "ring", "mean_P0_fractional_error",
    "mean_mbar_error_mag", "se_mean_mbar_error_mag",
    "mean_delta_mbar_vs_no_winsor_mag",
    "se_delta_mbar_vs_no_winsor_mag",
    "mean_changed_fraction",
]].round(5))
display(Image(filename=str(recovery_figure)))

## 9. Карта сохранённых продуктов

Все продукты лежат внутри проекта. Production-каталог `runs/sbf2_go3055` не изменяется.

In [ ]:
artifacts = []
for result in results:
    artifacts.append({
        "galaxy": result["galaxy"],
        "kind": "full normalized residual",
        "ring": "full frame",
        "branch": result["candidate_branch"],
        "fits": result["full_normalized_residual_fits"],
    })
    for item in result["normalized_fits"]:
        artifacts.append({
            "galaxy": result["galaxy"],
            "kind": "optional ring FFT input",
            "ring": item["ring"],
            "branch": item["branch"],
            "fits": item["path"],
        })
display(pd.DataFrame(artifacts))